# R²D-HOPE-MoRE — Knowledge Distillation from OpenRouter Teachers

This notebook distills reasoning capabilities from large teacher models (100B+ params)
into R²D-HOPE-MoRE (6-11M params) using Chain-of-Thought traces.

**How it works:**
1. Generate reasoning traces from OpenRouter teachers (Kimi, Qwen, DeepSeek)
2. Cache responses to avoid API re-calls
3. Fine-tune R²D-HOPE-MoRE on teacher reasoning patterns via diffusion objective

**Teacher Recommendations:**
- **Coding**: `kimi-k2.5` — best for code + explanation traces
- **General**: `qwen-122b` — balanced cost/quality
- **Maximum quality**: `qwen-397b` — use sparingly (expensive)
- **Step-by-step rigor**: `deepseek-v3` — explicit reasoning structure

**Cost Estimate:** ~$50-150 for 100K distillation samples

**Expected Gains:**
- WikiText PPL: modest improvement (+10%)
- Math reasoning (GSM8K): 5% → 25-40%
- Code generation: 2% → 15-25%

**Prerequisite:** Pre-trained R²D-HOPE-MoRE checkpoint from TRAIN_COLAB.ipynb

## Step 0 — Setup: Mount Drive, Install, Load Package

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/r2d_hope_training'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Install dependencies
!pip install -q torch transformers datasets tokenizers accelerate
!pip install -q requests  # for OpenRouter API

# Clone repo
!git clone https://github.com/TracNg99/R2D-HOPE-MoRE.git /content/R2D-HOPE-MoRE
sys.path.insert(0, '/content/R2D-HOPE-MoRE')

from r2d_hope import R2DConfig, R2D_HOPE_MoRE, build_or_load_tokenizer
from r2d_hope import OpenRouterDistiller, TEACHER_MODELS, DistillationTrainer, DistillConfig
print('✓ Setup complete')

## Step 1 — Store OpenRouter API Key

Get your API key from https://openrouter.ai/keys

**Security best practice:** Use Colab Secrets (left sidebar → key icon) rather than hardcoding:
- Secret name: `OPENROUTER_API_KEY`
- Value: `sk-or-v1-...`

The code below tries Colab Secrets first, falls back to environment variable.

In [ ]:
def get_openrouter_key():
    try:
        from google.colab import userdata
        return userdata.get('OPENROUTER_API_KEY')
    except:
        return os.environ.get('OPENROUTER_API_KEY')

API_KEY = get_openrouter_key()
if not API_KEY:
    print('⚠ OPENROUTER_API_KEY not found. Please set it in Colab Secrets or environment.')
else:
    print(f'✓ API key loaded: {API_KEY[:15]}...')

## Phase 1 — Generate Reasoning Traces from Teachers

Choose your domain and generate high-quality CoT data.

**Quick Test (100 samples, ~$1-3):**
- Use this to validate the pipeline before scaling

**Full Dataset (10K-100K samples, ~$50-150):**
- Multiple teachers, self-consistency voting, diverse domains

In [ ]:
from r2d_hope.distillation_data import get_reasoning_prompts

# ── Choose your domain ───────────────────────────────────────────────────────
DOMAIN = 'math'  # 'math' | 'code' | 'logic' | 'reading' | 'qa'

# Sample prompts for testing (replace with your actual dataset)
if DOMAIN == 'math':
    items = [
        'Calculate 127 × 34 and explain your steps',
        'What is the area of a circle with radius 5? Show work.',
        'Solve: 3x + 7 = 22',
        'A rectangle has length 12 and width 8. Find perimeter.',
        'If 5 workers complete a job in 8 days, how long for 10 workers?',
        'What is 15% of 240?',
        'Find the square root of 144 step by step.',
        'Calculate compound interest: $1000 at 5% for 3 years.',
        'Solve the system: x + y = 10, x - y = 4',
        'What is the sum of angles in a hexagon?',
    ]
elif DOMAIN == 'code':
    items = [
        'Write a function to reverse a string in Python. Explain your approach.',
        'Implement binary search with explanation.',
        'Create a function to check if a number is prime.',
        'Write code to find the longest common subsequence.',
        'Implement quicksort with detailed comments.',
    ]
else:
    items = [
        'Explain why the sky appears blue.',
        'What causes seasons on Earth? Explain step by step.',
        'How does photosynthesis work?',
        'Why do we have leap years?',
        'Explain the water cycle.',
    ]

prompts = get_reasoning_prompts(DOMAIN, items)
print(f'Generated {len(prompts)} prompts for domain: {DOMAIN}')
for p in prompts[:3]:
    print(f'  • {p[:60]}...')

In [ ]:
# ── Generate from teachers ───────────────────────────────────────────────────
TEACHERS = ['qwen-122b']  # Start with cost-effective teacher
# For higher quality: ['qwen-122b', 'deepseek-v3'] or ['qwen-397b']

distiller = OpenRouterDistiller(api_key=API_KEY, cache_dir=f'{DRIVE_ROOT}/distill_cache')

output_path = f'{DRIVE_ROOT}/distillation_data_{DOMAIN}.jsonl'

samples = distiller.generate_dataset(
    prompts=prompts,
    teachers=[TEACHER_MODELS[t] for t in TEACHERS],
    output_path=output_path,
    n_samples_per_prompt=1,  # Increase to 3-5 for self-consistency voting
    use_consistency=False,
)

print(f'\n✓ Generated {len(samples)} samples')
print(f'  Saved to: {output_path}')
print(f'  Cache: {DRIVE_ROOT}/distill_cache')

### Inspect Generated Samples

In [ ]:
import json

with open(output_path, 'r') as f:
    all_samples = [json.loads(line) for line in f]

print(f'Total samples: {len(all_samples)}')
print('\n' + '=' * 60)
print('Sample 1:')
print('=' * 60)
s = all_samples[0]
print(f'Prompt: {s["prompt_text"][:100]}...')
print(f'\nReasoning ({len(s["reasoning_ids"])} tokens):')
print(s['reasoning_text'][:300] + '...' if len(s['reasoning_text']) > 300 else s['reasoning_text'])
print(f'\nTeachers: {s["teacher_votes"]}')

## Phase 2 — Load Pre-trained Model

Load your pre-trained checkpoint from TRAIN_COLAB.ipynb.
Distillation builds on this foundation — the model already knows language,
now it learns to reason like the teacher.

In [ ]:
import torch
import glob

# ── Model configuration ────────────────────────────────────────────────────
# Must match the config used in TRAIN_COLAB.ipynb
MODEL_SIZE = 'small'  # 'small' (6.5M) | 'medium' (11.5M)

if MODEL_SIZE == 'small':
    model_cfg = R2DConfig(
        d_model=384, d_ffn=1024, d_embedding=192, num_heads=6, head_dim=64,
        vocab_size=16384, nested_depth=20, num_experts=4, top_k_experts=2,
        window_size=128, ddim_inference_steps=10,
    )
else:
    model_cfg = R2DConfig(
        d_model=512, d_ffn=1280, d_embedding=256, num_heads=8, head_dim=64,
        vocab_size=16384, nested_depth=20, num_experts=4, top_k_experts=2,
        window_size=128, ddim_inference_steps=10,
    )

model = R2D_HOPE_MoRE(model_cfg)

# Load latest checkpoint
ckpt_dir = f'{DRIVE_ROOT}/checkpoints/r2d_hope_{MODEL_SIZE}'
ckpts = sorted(glob.glob(f'{ckpt_dir}/step_*.pt'))

if ckpts:
    ckpt = torch.load(ckpts[-1], map_location='cpu')
    model.load_state_dict(ckpt['model'])
    print(f'✓ Loaded checkpoint: {ckpts[-1]}')
    print(f'  Pre-training step: {ckpt.get("step", "unknown")}')
else:
    print('⚠ No checkpoint found — starting from random init')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'✓ Model on {device}')

## Phase 3 — Distillation Training

Fine-tune the model on teacher reasoning traces.

**Key hyperparameters:**
- `ce_loss_weight`: Higher than pre-training (0.5 vs 0.1) — we prioritize imitating teacher
- `max_steps`: 5K-10K typically enough for fine-tuning
- `lr`: Lower than pre-training (2e-4) — gentle updates to pre-trained weights

In [ ]:
from r2d_hope.trainer import TrainConfig

# ── Distillation config ────────────────────────────────────────────────────
distill_cfg = DistillConfig(
    data_path=output_path,
    max_seq_len=512,
    noise_loss_weight=1.0,     # Keep diffusion objective
    ce_loss_weight=0.5,        # Higher than pre-training — imitate teacher!
    distill_loss_weight=1.0,   # Additional distillation loss
    aux_loss_weight=0.01,      # MoE balancing
    batch_size=8,
    grad_accum_steps=4,       # Effective batch = 32
    max_steps=5_000,           # Fine-tuning needs fewer steps
    warmup_steps=500,
    lr=2e-4,                  # Lower than pre-training
    lr_min=2e-5,
    eval_every=500,
    save_every=1_000,
)

# Base training config (for checkpointing paths, etc.)
train_cfg = TrainConfig(
    run_name=f'r2d_hope_{MODEL_SIZE}_distilled_{DOMAIN}',
    checkpoint_dir=f'{DRIVE_ROOT}/checkpoints',
    log_dir=f'{DRIVE_ROOT}/logs',
    tokenizer_dir=f'{DRIVE_ROOT}/tokenizer',
)

print('Distillation configuration:')
print(f'  Domain: {DOMAIN}')
print(f'  Data: {distill_cfg.data_path}')
print(f'  Steps: {distill_cfg.max_steps}')
print(f'  Batch: {distill_cfg.batch_size}×{distill_cfg.grad_accum_steps}')
print(f'  CE weight: {distill_cfg.ce_loss_weight} (vs 0.1 in pre-training)')

In [ ]:
# ── Run distillation ───────────────────────────────────────────────────────
trainer = DistillationTrainer(model, distill_cfg, train_cfg)
history = trainer.train()

print('\n✓ Distillation complete!')
print(f'Final checkpoint saved to {train_cfg.checkpoint_dir}')

## Phase 4 — Evaluate Distilled Model

Compare before/after on reasoning tasks.

In [ ]:
# Load pre-trained baseline for comparison
baseline_model = R2D_HOPE_MoRE(model_cfg).to(device)
if ckpts:
    baseline_ckpt = torch.load(ckpts[-1], map_location=device)
    baseline_model.load_state_dict(baseline_ckpt['model'])
    baseline_model.eval()

model.eval()

# Test prompts
test_prompts = [
    'What is 15 × 24? Show your work.',
    'Explain how rain is formed step by step.',
    'If a car travels 60 km/h for 2.5 hours, how far does it go?',
]

tokenizer = build_or_load_tokenizer(
    tokenizer_dir=train_cfg.tokenizer_dir,
    vocab_size=model_cfg.vocab_size,
)

print('=' * 60)
print('Comparison: Baseline vs Distilled')
print('=' * 60)

for prompt in test_prompts:
    print(f'\nPrompt: {prompt}')
    
    # Encode
    prompt_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    ctx = torch.zeros(1, 1, model_cfg.d_model, device=device)
    
    # Baseline generation
    with torch.no_grad():
        baseline_out = baseline_model.generate(
            prompt_ids, ctx, num_answer_tokens=64, ddim_steps=10
        )
    baseline_text = tokenizer.decode(baseline_out[0], skip_special_tokens=True)
    
    # Distilled generation
    with torch.no_grad():
        distilled_out = model.generate(
            prompt_ids, ctx, num_answer_tokens=64, ddim_steps=10
        )
    distilled_text = tokenizer.decode(distilled_out[0], skip_special_tokens=True)
    
    print(f'Baseline:  {baseline_text[:100]}...')
    print(f'Distilled: {distilled_text[:100]}...')
    print('-' * 40)

## Phase 5 — Benchmark Against Teachers

Run the same prompts through the original teachers for comparison.
Note: This uses API credits — run sparingly.

In [ ]:
# Compare one sample against teacher
test_prompt = test_prompts[0]

print(f'Test prompt: {test_prompt}')
print('\nFetching teacher responses (using cache if available)...')

for teacher_key in ['qwen-122b', 'deepseek-v3']:
    try:
        trace = distiller.generate_trace(
            test_prompt, 
            TEACHER_MODELS[teacher_key],
            use_cache=True
        )
        print(f'\n{teacher_key}:')
        print(f'  {trace.reasoning[:150]}...')
        print(f'  Answer: {trace.answer[:50]}')
    except Exception as e:
        print(f'{teacher_key}: Error - {e}')

## Final Save

In [ ]:
final_path = f'{DRIVE_ROOT}/final_{MODEL_SIZE}_distilled_{DOMAIN}.pt'
torch.save({
    'model_state': model.state_dict(),
    'model_cfg': vars(model_cfg),
    'distill_cfg': vars(distill_cfg),
    'domain': DOMAIN,
}, final_path)
print(f'✓ Saved distilled model to {final_path}')

## Next Steps

1. **Scale up**: Generate 10K-100K samples for full distillation
2. **Multi-teacher ensemble**: Combine responses from multiple teachers
3. **DPO training**: Use preference pairs for fine-grained alignment
4. **Evaluate on benchmarks**: GSM8K, HumanEval, etc.

**Cost-conscious scaling:**
- Start with 1K samples (~$5) to validate gains
- If successful, scale to 10K (~$50) or 100K (~$150)
- Use `qwen-122b` for 80% of data, `qwen-397b` for hard 20%